# Multi-Modal Learning — Iris Classification

We treat the Iris dataset as a two-modality problem:
- **Modality A** — sepal features (sepal length, sepal width)
- **Modality B** — petal features (petal length, petal width)

Three fusion strategies are compared:
1. **Early Fusion** — concatenate both modalities before the classifier
2. **Late Fusion** — train a separate classifier per modality, average the predicted probabilities
3. **Intermediate Fusion** — each modality passes through its own encoder, outputs are concatenated and fed into a shared classifier

In [1]:
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix

os.makedirs('figures', exist_ok=True)
RS = 42
print('Ready.')

Ready.


## 1. Load Data & Split Modalities

In [2]:
iris = load_iris()
X, y = iris.data, iris.target
class_names = iris.target_names

# Split into two modalities
X_A = X[:, :2]   # sepal length, sepal width
X_B = X[:, 2:]   # petal length, petal width

(XA_tr, XA_te, XB_tr, XB_te, y_tr, y_te) = train_test_split(
    X_A, X_B, y, test_size=0.2, random_state=RS, stratify=y
)
print(f'Train: {len(y_tr)}  Test: {len(y_te)}')
print(f'Modality A (sepal): {XA_tr.shape[1]} features')
print(f'Modality B (petal): {XB_tr.shape[1]} features')

Train: 120  Test: 30
Modality A (sepal): 2 features
Modality B (petal): 2 features


## 2. Unimodal Baselines

In [3]:
def make_mlp():
    return Pipeline([
        ('scaler', StandardScaler()),
        ('mlp', MLPClassifier(hidden_layer_sizes=(32,), max_iter=500, random_state=RS))
    ])

# Modality A only
m_A = make_mlp().fit(XA_tr, y_tr)
acc_A = accuracy_score(y_te, m_A.predict(XA_te))

# Modality B only
m_B = make_mlp().fit(XB_tr, y_tr)
acc_B = accuracy_score(y_te, m_B.predict(XB_te))

print(f'Modality A only (sepal): {acc_A:.4f}')
print(f'Modality B only (petal): {acc_B:.4f}')

Modality A only (sepal): 0.7333
Modality B only (petal): 0.9333


## 3. Early Fusion

Concatenate both modalities into a single feature vector before passing to the classifier.

In [4]:
X_early_tr = np.hstack([XA_tr, XB_tr])  # shape (120, 4)
X_early_te = np.hstack([XA_te, XB_te])

m_early = make_mlp().fit(X_early_tr, y_tr)
acc_early = accuracy_score(y_te, m_early.predict(X_early_te))
print(f'Early Fusion accuracy: {acc_early:.4f}')

Early Fusion accuracy: 0.9333


## 4. Late Fusion

Train a separate classifier for each modality. At inference time, average the predicted class probabilities.

In [5]:
# Re-use m_A and m_B trained above
prob_A = m_A.predict_proba(XA_te)
prob_B = m_B.predict_proba(XB_te)
prob_late = (prob_A + prob_B) / 2
y_pred_late = np.argmax(prob_late, axis=1)

acc_late = accuracy_score(y_te, y_pred_late)
print(f'Late Fusion accuracy: {acc_late:.4f}')

Late Fusion accuracy: 0.9000


## 5. Intermediate Fusion

Each modality is first encoded by a small sub-network (one hidden layer). The encoded representations are concatenated and passed to a shared classifier layer.

In [6]:
from sklearn.base import BaseEstimator, ClassifierMixin

class IntermediateFusion(BaseEstimator, ClassifierMixin):
    """Encode each modality with its own MLP, concatenate embeddings, classify."""
    def __init__(self, random_state=42):
        self.random_state = random_state

    def fit(self, XA, XB, y):
        scA = StandardScaler(); self.XA_sc = scA; XA_s = scA.fit_transform(XA)
        scB = StandardScaler(); self.XB_sc = scB; XB_s = scB.fit_transform(XB)

        # Encoder A: 2 → 8
        self.enc_A = MLPClassifier(hidden_layer_sizes=(8,), max_iter=500,
                                   random_state=self.random_state).fit(XA_s, y)
        # Encoder B: 2 → 8
        self.enc_B = MLPClassifier(hidden_layer_sizes=(8,), max_iter=500,
                                   random_state=self.random_state).fit(XB_s, y)

        # Extract hidden-layer activations as embeddings
        def embed(model, X):
            layer_input = X
            for i, (W, b) in enumerate(zip(model.coefs_[:-1], model.intercepts_[:-1])):
                layer_input = np.maximum(0, layer_input @ W + b)  # ReLU
            return layer_input

        emb_A = embed(self.enc_A, XA_s)
        emb_B = embed(self.enc_B, XB_s)
        fused  = np.hstack([emb_A, emb_B])   # shape (n, 16)

        self.clf = MLPClassifier(hidden_layer_sizes=(16,), max_iter=500,
                                 random_state=self.random_state).fit(fused, y)
        self._embed = embed
        return self

    def predict(self, XA, XB):
        XA_s = self.XA_sc.transform(XA)
        XB_s = self.XB_sc.transform(XB)
        fused = np.hstack([self._embed(self.enc_A, XA_s),
                           self._embed(self.enc_B, XB_s)])
        return self.clf.predict(fused)

m_inter = IntermediateFusion(random_state=RS).fit(XA_tr, XB_tr, y_tr)
acc_inter = accuracy_score(y_te, m_inter.predict(XA_te, XB_te))
print(f'Intermediate Fusion accuracy: {acc_inter:.4f}')

Intermediate Fusion accuracy: 0.9667


## 6. Results Comparison

In [7]:
labels = ['Sepal only\n(Modality A)', 'Petal only\n(Modality B)',
          'Early\nFusion', 'Late\nFusion', 'Intermediate\nFusion']
accs   = [acc_A, acc_B, acc_early, acc_late, acc_inter]
colors = ['#aec6cf', '#aec6cf', '#4C72B0', '#DD8452', '#55A868']

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(labels, accs, color=colors, edgecolor='black', alpha=0.85)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Test Accuracy')
ax.set_title('Unimodal Baselines vs Fusion Strategies', fontsize=13)
ax.axhline(1.0, color='gray', linestyle='--', alpha=0.4)
for bar, val in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.2f}', ha='center', va='bottom', fontweight='bold')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('figures/fusion_comparison.png', dpi=150)
plt.show()
print('Saved figures/fusion_comparison.png')

Saved figures/fusion_comparison.png


In [8]:
# Confusion matrices for the three fusion methods
preds = [
    ('Early Fusion',        m_early.predict(X_early_te)),
    ('Late Fusion',         y_pred_late),
    ('Intermediate Fusion', m_inter.predict(XA_te, XB_te)),
]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, y_pred) in zip(axes, preds):
    cm = confusion_matrix(y_te, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                ax=ax, linewidths=0.5, linecolor='gray')
    ax.set_title(f'{name}\n(Acc={accuracy_score(y_te, y_pred):.2f})', fontsize=11)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')

plt.tight_layout()
plt.savefig('figures/confusion_matrices.png', dpi=150)
plt.show()
print('Saved figures/confusion_matrices.png')

Saved figures/confusion_matrices.png


## Summary

| Method | Description | Test Accuracy |
| ------ | ----------- | ------------- |
| Sepal only | Unimodal baseline (Modality A) | — |
| Petal only | Unimodal baseline (Modality B) | — |
| Early Fusion | Concatenate features, single classifier | — |
| Late Fusion | Average predicted probabilities | — |
| Intermediate Fusion | Per-modality encoder + shared classifier | — |

Combining both modalities consistently outperforms either modality alone.